# Hardware Benchmark — FPGA ZCU102

Publication-ready latency and energy figures for the manuscript (§5.4 Efficiency).

Two cuts:
1. **4-arch FPGA-only** (`s1`) — latency breakdown + energy/patch across all four architectures.
2. **3-platform** (`CPU / GPU / FPGA-s1`) — FP and ResSH only, compress scenario.

All data from `results/benchmark_hardware/` + `results/benchmark_unified/`.
Regenerate with `python scripts/benchmark/run_unified_benchmark.py --model-dir results/fpga/active_model/ --power`.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rootutils
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

ROOT = rootutils.setup_root(Path.cwd(), indicator=".project-root", pythonpath=True)
sys.path.insert(0, str(ROOT / "notebooks"))

from _benchmark_loader import load_runs, load_stage_breakdowns
from _plotkit import PALETTE, ARCH_LABEL, ARCH_ORDER as _ARCH_ORDER

FIG_DPI = 120
PLOTS_DIR = ROOT / "results" / "plots" / "hardware_benchmark"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
MANUSCRIPT_DIR = ROOT / "LaTeX" / "SAR_DDC_FPGA_TGRS_2026" / "figures" / "images"
SAVE_FIGURES = True

PLAT = PALETTE["platforms"]
STAGE_CLR = PALETTE["cpp_stages"]

# Arch order for this notebook (FPGA-natural: FP → SH → ResFP → ResSH by complexity)
ARCH_ORDER = ["FP", "SHyp", "ResFP", "ResSHyp"]

# Stage stacking order
STAGE_ORDER = [
    "normalize",
    "g_a",
    "h_a",
    "eb_compress",
    "eb_decompress",
    "h_s",
    "gc_compress",
    "gc_decompress",
    "g_s",
    "denorm",
]
STAGE_DISPLAY = {"host_concat_abs": "concat_abs", "host_split_y_hat": "split_y_hat"}

# ── SERIES ────────────────────────────────────────────────────────────────────
SERIES = {
    "CPU": dict(
        platform="cpu", config="baseline", color=PLAT["cpu_dynamic"], hatch="", label="CPU"
    ),
    "GPU": dict(
        platform="gpu", config="baseline", color=PLAT["gpu_dynamic"], hatch="", label="GPU"
    ),
    "FPGA": dict(platform="fpga", config="s1", color=PLAT["fpga_dynamic"], hatch="", label="FPGA"),
}

# ── Load data ─────────────────────────────────────────────────────────────────
df = load_runs(ROOT / "results/benchmark_hardware", ROOT / "results/benchmark_unified")
sb = load_stage_breakdowns(ROOT / "results/benchmark_hardware", ROOT / "results/benchmark_unified")
df["edp_mJ_ms"] = df["energy_mJ_per_patch"] * df["total_latency_mean_ms"]


def _row(arch, spec, scenario):
    r = df[
        (df.arch == arch)
        & (df.platform == spec["platform"])
        & (df.config == spec["config"])
        & (df.scenario == scenario)
        & (df.model_name.str.contains("_L1000"))
    ]
    return r.iloc[0] if not r.empty else None


print(f"Loaded {len(df)} benchmark rows from {sorted(df.platform.unique())}")

## §1 · 4-arch FPGA-only latency breakdown

Per-stage latency breakdown for all four architectures on FPGA (s1, compress scenario).
Same as the F6 figure in `benchmark_cross_platform_analysis.ipynb` but FPGA-only and all 4 archs.

In [ ]:
def stage_breakdown_fpga(
    archs=None,
    config="s1",
    scenario="compress",
    pct_labels=True,
    pct_min=0.06,
    save=None,
):
    """Stacked per-stage latency, one bar per arch, FPGA only (s0 or s1)."""
    archs = archs or [a for a in ARCH_ORDER if a in df.arch.values]
    fig, ax = plt.subplots(figsize=(2.0 * len(archs) + 1.5, 5), dpi=FIG_DPI)
    ax.set_axisbelow(True)
    ax.yaxis.grid(True, lw=0.4, alpha=0.3, color="#aaa")
    present = []
    for xi, arch in enumerate(archs):
        spec = {"platform": "fpga", "config": config}
        r = _row(arch, spec, scenario)
        if r is None:
            continue
        st = sb[
            (sb.platform == "fpga")
            & (sb.model_name == r.model_name)
            & (sb.config == config)
            & (sb.scenario == scenario)
        ]
        total = st.mean_ms.sum()
        bottom = 0.0
        for stg in STAGE_ORDER:
            row_st = st[st.stage == stg]
            if row_st.empty:
                continue
            h = row_st.mean_ms.iloc[0]
            ax.bar(
                xi,
                h,
                bottom=bottom,
                width=0.75,
                color=STAGE_CLR.get(stg, "#999"),
                edgecolor="white",
                lw=0.3,
                zorder=3,
            )
            if pct_labels and total and h / total >= pct_min:
                ax.text(
                    xi,
                    bottom + h / 2,
                    f"{100 * h / total:.0f}%",
                    ha="center",
                    va="center",
                    fontsize=7,
                    color="white",
                    fontweight="bold",
                )
            if stg not in present:
                present.append(stg)
            bottom += h
        if bottom:
            ax.text(xi, bottom, f"{bottom:.0f}", ha="center", va="bottom", fontsize=8.5)
    ax.set_xticks(range(len(archs)))
    ax.set_xticklabels([ARCH_LABEL.get(a, a) for a in archs], fontsize=11)
    ax.set_ylabel("latency / patch (ms)")
    handles = [
        Patch(facecolor=STAGE_CLR.get(s, "#999"), label=STAGE_DISPLAY.get(s, s)) for s in present
    ]
    fig.legend(
        handles=handles,
        loc="lower center",
        ncol=min(len(handles), 8),
        fontsize=7.5,
        bbox_to_anchor=(0.5, 0.0),
    )
    fig.tight_layout(rect=(0, 0.06, 1, 1))
    if save:
        fig.savefig(PLOTS_DIR / f"{save}.pdf", bbox_inches="tight")
        print(f"Saved: {PLOTS_DIR / save}.pdf")
    plt.show()


# ── 4-arch FPGA-only (s1, compress) ──────────────────────────────────────────
stage_breakdown_fpga(save="fig_latency_fpga_4arch")

## §2 · Energy per patch — FPGA only (4 arch)

Total energy/patch (mJ) on FPGA-s1 across all four architectures.

In [ ]:
def energy_bars_fpga(archs=None, config="s1", scenario="compress", save=None):
    """Energy/patch bars for FPGA only (4 archs)."""
    archs = archs or [a for a in ARCH_ORDER if a in df.arch.values]
    spec = {"platform": "fpga", "config": config}
    vals = {a: _row(a, spec, scenario) for a in archs}
    vals = {a: r["energy_mJ_per_patch"] for a, r in vals.items() if r is not None}
    ymax = max(vals.values(), default=1)
    fig, ax = plt.subplots(figsize=(2.0 * len(archs) + 1.5, 4.5), dpi=FIG_DPI)
    ax.set_axisbelow(True)
    ax.yaxis.grid(True, lw=0.4, alpha=0.3, color="#aaa")
    for xi, arch in enumerate(archs):
        v = vals.get(arch)
        if v is None:
            continue
        ax.bar(xi, v, width=0.7, color=PLAT["fpga_dynamic"], edgecolor="white", lw=0.5, zorder=3)
        ax.text(xi, v + ymax * 0.012, f"{v:.1f}", ha="center", va="bottom", fontsize=10)
    ax.set_xticks(range(len(archs)))
    ax.set_xticklabels([ARCH_LABEL.get(a, a) for a in archs], fontsize=11)
    ax.set_ylabel("energy / patch (mJ)")
    ax.set_ylim(0, ymax * 1.18)
    fig.tight_layout()
    if save:
        fig.savefig(PLOTS_DIR / f"{save}.pdf", bbox_inches="tight")
        print(f"Saved: {PLOTS_DIR / save}.pdf")
    plt.show()


energy_bars_fpga(save="fig_energy_fpga_4arch")

## §3 · 3-platform comparison — FP and ResSH

Latency and energy/patch across CPU / GPU / FPGA for the two anchor architectures
(lightest = FP, heaviest = ResSHyp), compress scenario.

In [ ]:
# Reuse the full stage_breakdown + grouped_bars logic from benchmark_cross_platform_analysis.ipynb
# but restricted to ["FP", "ResSHyp"] and ["CPU", "GPU", "FPGA"].
ARCHS_3P = ["FP", "ResSHyp"]
SERIES_3P = ["CPU", "GPU", "FPGA"]


def grouped_bars_3p(
    metric, ylabel, archs=ARCHS_3P, series=SERIES_3P, scenario="compress", ref="CPU", save=None
):
    """Grouped bar chart — 3 platforms × 2 archs."""
    x = np.arange(len(archs))
    n = len(series)
    w = 0.8 / n
    vals = {
        (a, s): (
            _row(a, SERIES[s], scenario)[metric]
            if _row(a, SERIES[s], scenario) is not None
            and not pd.isna(_row(a, SERIES[s], scenario)[metric])
            else None
        )
        for a in archs
        for s in series
    }
    ymax = max((v for v in vals.values() if v is not None), default=1)
    fig, ax = plt.subplots(figsize=(3.0 * len(archs) + 1.5, 5), dpi=FIG_DPI)
    ax.set_axisbelow(True)
    ax.yaxis.grid(True, lw=0.4, alpha=0.35, color="#aaa")
    for ai, arch in enumerate(archs):
        refv = vals.get((arch, ref))
        for si, s in enumerate(series):
            v = vals.get((arch, s))
            if v is None:
                continue
            xp = x[ai] + (si - (n - 1) / 2) * w
            spec = SERIES[s]
            ax.bar(xp, v, width=w, color=spec["color"], edgecolor="white", lw=0.5, zorder=3)
            ax.text(
                xp,
                v + ymax * 0.012,
                f"{v:.0f}" if v >= 10 else f"{v:.1f}",
                ha="center",
                va="bottom",
                fontsize=9,
            )
            if s != ref and refv:
                fac = (refv / v) if True else (v / refv)  # lower_is_better=True
                ax.text(
                    xp,
                    v + ymax * 0.065,
                    f"{fac:.0f}×" if fac >= 10 else f"{fac:.1f}×",
                    ha="center",
                    va="bottom",
                    fontsize=8,
                    fontweight="bold",
                    color="#222",
                )
    ax.set_xticks(x)
    ax.set_xticklabels([ARCH_LABEL.get(a, a) for a in archs], fontsize=11)
    ax.set_ylabel(ylabel)
    ax.set_ylim(0, ymax * 1.22)
    ax.legend(
        handles=[
            Patch(facecolor=SERIES[s]["color"], edgecolor="white", label=SERIES[s]["label"])
            for s in series
        ],
        fontsize=10,
        framealpha=0.9,
    )
    fig.tight_layout()
    if save:
        fig.savefig(PLOTS_DIR / f"{save}.pdf", bbox_inches="tight")
        print(f"Saved: {PLOTS_DIR / save}.pdf")
    plt.show()


grouped_bars_3p("total_latency_mean_ms", "latency / patch (ms)", save="fig_latency_3platform")
grouped_bars_3p("energy_mJ_per_patch", "energy / patch (mJ)", save="fig_energy_3platform")

## §4 · Summary tables — latency and energy/patch

Clean numeric tables (text + LaTeX) for the manuscript's §5.4 numbers.

- **Table A** — 4-arch FPGA-only (s1, compress): absolute latency and energy/patch.
- **Table B** — 3-platform (FP + ResSH, compress): absolute values + speedup / efficiency vs CPU.

In [ ]:
def _fv(v, fmt=".1f"):
    """Format a value, returning 'N/A' for NaN/None."""
    return "N/A" if pd.isna(v) else format(float(v), fmt)


def _fx(v, baseline=False):
    """Format a speedup/efficiency factor; '—' for the baseline platform."""
    if baseline:
        return "—"
    return "N/A" if pd.isna(v) else f"{float(v):.1f}×"


SCENARIO_TBL = "compress"

# ── Table A: 4-arch FPGA-only (s1, compress) ──────────────────────────────────
print("Table A — FPGA-only  (s1, compress)\n")
print(f"{'Arch':>8}  {'Latency [ms]':>14}  {'Energy/patch [mJ]':>18}")
print("-" * 46)
fpga_rows = {}
for arch in ARCH_ORDER:
    r = _row(arch, {"platform": "fpga", "config": "s1"}, SCENARIO_TBL)
    if r is None:
        print(f"{ARCH_LABEL.get(arch, arch):>8}  (no data)")
        continue
    fpga_rows[arch] = r
    print(
        f"{ARCH_LABEL.get(arch, arch):>8}  {_fv(r.total_latency_mean_ms):>14}  "
        f"{_fv(r.energy_mJ_per_patch):>18}"
    )

# ── Table B: 3-platform (FP + ResSH, compress) ────────────────────────────────
print("\n\nTable B — 3-platform  (compress)\n")
print(
    f"{'Arch':>7}  {'Platform':>8}  {'Latency [ms]':>13}  {'vs CPU':>8}  "
    f"{'Energy [mJ]':>12}  {'vs CPU':>8}"
)
print("-" * 68)
for arch in ARCHS_3P:
    cpu_r = _row(arch, SERIES["CPU"], SCENARIO_TBL)
    cpu_lat = cpu_r.total_latency_mean_ms if cpu_r is not None else float("nan")
    cpu_nrg = cpu_r.energy_mJ_per_patch if cpu_r is not None else float("nan")
    for s_name in SERIES_3P:
        r = _row(arch, SERIES[s_name], SCENARIO_TBL)
        if r is None:
            continue
        lat = r.total_latency_mean_ms
        nrg = r.energy_mJ_per_patch
        spd = cpu_lat / lat if not pd.isna(lat) and lat > 0 else float("nan")
        eff = cpu_nrg / nrg if not pd.isna(nrg) and nrg > 0 else float("nan")
        print(
            f"{ARCH_LABEL.get(arch, arch):>7}  {s_name:>8}  {_fv(lat):>13}  "
            f"{_fx(spd, s_name == 'CPU'):>8}  {_fv(nrg):>12}  {_fx(eff, s_name == 'CPU'):>8}"
        )

# ── LaTeX — Table A ────────────────────────────────────────────────────────────
print("\n\n% LaTeX — Table A (FPGA-only, 4 arch)\n")
_la = [
    r"\begin{tabular}{lrr}",
    r"\toprule",
    r"Architecture & Latency [ms] & Energy/patch [mJ] \\",
    r"\midrule",
]
for arch in ARCH_ORDER:
    r = fpga_rows.get(arch)
    if r is None:
        continue
    _la.append(
        f"{ARCH_LABEL.get(arch, arch)} & {_fv(r.total_latency_mean_ms)} & "
        f"{_fv(r.energy_mJ_per_patch)} \\\\"
    )
_la += [r"\bottomrule", r"\end{tabular}"]
print("\n".join(_la))

# ── LaTeX — Table B ────────────────────────────────────────────────────────────
print("\n\n% LaTeX — Table B (3-platform)\n")
_lb = [
    r"\begin{tabular}{llrrrr}",
    r"\toprule",
    r"Architecture & Platform & Latency [ms] & Speedup & Energy [mJ] & Efficiency \\",
    r"\midrule",
]
for arch in ARCHS_3P:
    cpu_r = _row(arch, SERIES["CPU"], SCENARIO_TBL)
    cpu_lat = cpu_r.total_latency_mean_ms if cpu_r is not None else float("nan")
    cpu_nrg = cpu_r.energy_mJ_per_patch if cpu_r is not None else float("nan")
    for s_name in SERIES_3P:
        r = _row(arch, SERIES[s_name], SCENARIO_TBL)
        if r is None:
            continue
        lat = r.total_latency_mean_ms
        nrg = r.energy_mJ_per_patch
        spd = cpu_lat / lat if not pd.isna(lat) and lat > 0 else float("nan")
        eff = cpu_nrg / nrg if not pd.isna(nrg) and nrg > 0 else float("nan")
        spd_str = r"\textemdash" if s_name == "CPU" else f"${_fv(spd)}\\times$"
        eff_str = r"\textemdash" if s_name == "CPU" else f"${_fv(eff)}\\times$"
        _lb.append(
            f"{ARCH_LABEL.get(arch, arch)} & {s_name} & {_fv(lat)} & "
            f"{spd_str} & {_fv(nrg)} & {eff_str} \\\\"
        )
_lb += [r"\bottomrule", r"\end{tabular}"]
print("\n".join(_lb))